New ML test pipeline script


Import all libraries needed for this ML script.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)
print('Imports Successfully!')

Imports Successfully!


In [3]:
#to check the python version and the path of the python executable
import sys
print(sys.executable)

/Users/boracomert/Desktop/Osint_project/.venv/bin/python


the next step: Load data 

In [4]:
DATA_PATH = "../data/processed/labeled_merged_data_cleaned.csv"

df = pd.read_csv(DATA_PATH)
print('Data Loaded Successfully!')

#the model shouldnt see layoff information becuse it can cause data leakage, so we will drop the layoff columns from the features
META_COLS = ['company', 'date', 'quarter', 'layoff','same_quarter', 'next_quarter', 'layoff_same_quarter', 'layoff_next_quarter', 'layoff_same_or_next_quarter']

FEATURE_COLS = [col for col in df.columns if col not in META_COLS]


n_companies  = df['company'].nunique()
n_rows_pos   = len(df)

print(f'Positive companies : {n_companies}')
print(f'Positive rows      : {n_rows_pos}')
print(f'Feature columns    : {len(FEATURE_COLS)}')
print(f'Quarters present   : {sorted(df["quarter"].unique())}')
df.head(5)

Data Loaded Successfully!
Positive companies : 1941
Positive rows      : 12047
Feature columns    : 346
Quarters present   : ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2']


,company,date,quarter,bs_Ordinary Shares Number,bs_Share Issued,bs_Net Debt,bs_Total Debt,bs_Tangible Book Value,bs_Invested Capital,bs_Working Capital,...,fin_Other Non Interest Expense,fin_Depletion Income Statement,fin_Policyholder Benefits Ceded,fin_Net Income Extraordinary,same_quarter,next_quarter,layoff_same_quarter,layoff_next_quarter,layoff_same_or_next_quarter,layoff
0,AFCONS.BO,2024-09-30,2024Q3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024Q3,2024Q4,0,0,0,0
1,AFCONS.BO,2024-12-31,2024Q4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2024Q4,2025Q1,0,0,0,0
2,AFCONS.BO,2025-03-31,2025Q1,367784631.0,367784631.0,1.795550e+10,2.343300e+10,5.259830e+10,7.496240e+10,3.021220e+10,...,NaN,NaN,NaN,NaN,2025Q1,2025Q2,0,0,0,0
3,AFCONS.BO,2025-06-30,2025Q2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2025Q2,2025Q3,0,0,0,0
4,AFCONS.BO,2025-09-30,2025Q3,367784631.0,367784631.0,3.097260e+10,3.565230e+10,5.388340e+10,8.861110e+10,3.037610e+10,...,NaN,NaN,NaN,NaN,2025Q3,2025Q4,0,0,0,0


In [5]:
# there are many feature cols with almost all nAn values, we will drop those cols

threshold = 0.9
cols_to_drop = [col for col in FEATURE_COLS if df[col].isna().mean() > threshold]
print(f'Columns to drop due to high NaN percentage (> {threshold*100}%): {cols_to_drop}')
FEATURE_COLS = [col for col in FEATURE_COLS if col not in cols_to_drop]
print(f'Updated feature columns count: {len(FEATURE_COLS)}')



Columns to drop due to high NaN percentage (> 90.0%): ['bs_Non Current Accrued Expenses', 'bs_Current Deferred Taxes Liabilities', 'bs_Dividends Payable', 'bs_Non Current Prepaid Assets', 'bs_Financial Assets Designatedas Fair Value Through Profitor Loss Total', 'bs_Defined Pension Benefit', 'bs_Other Investments', 'bs_Financial Assets', 'bs_Investments In Other Ventures Under Equity Method', 'bs_Investmentsin Associatesat Cost', 'bs_Hedging Assets Current', 'bs_Foreign Currency Translation Adjustments', 'bs_Minimum Pension Liabilities', 'bs_Inventories Adjustments Allowances', 'bs_Receivables Adjustments Allowances', 'bs_Preferred Stock Equity', 'bs_Other Inventories', 'bs_Current Notes Payable', 'bs_Current Deferred Assets', 'bs_Investmentsin Joint Venturesat Cost', 'bs_Held To Maturity Securities', 'bs_Preferred Securities Outside Stock Equity', 'bs_Non Current Accounts Receivable', 'bs_Investment Properties', 'bs_Preferred Shares Number', 'bs_Loans Receivable', 'bs_Line Of Credit',

In [6]:

# Now we will check the correlation between the feature columns, feature columns with high multicolinearity can cause issues for some models,such as KNN 

corr_matrix = df[FEATURE_COLS].corr()
corr_matrix




,bs_Ordinary Shares Number,bs_Share Issued,bs_Net Debt,bs_Total Debt,bs_Tangible Book Value,bs_Invested Capital,bs_Working Capital,bs_Net Tangible Assets,bs_Capital Lease Obligations,bs_Common Stock Equity,...,fin_Interest Income Non Operating,fin_Research And Development,fin_Write Off,fin_Restructuring And Mergern Acquisition,fin_Depreciation Amortization Depletion Income Statement,fin_Amortization,fin_Other Special Charges,fin_Impairment Of Capital Assets,fin_Selling And Marketing Expense,fin_Total Other Finance Cost
bs_Ordinary Shares Number,1.000000,0.999833,0.563725,0.175143,0.088440,0.106398,0.120807,0.088417,0.055453,0.089080,...,0.074745,0.054518,0.613683,0.256032,0.904080,0.103441,0.295628,0.473271,0.099196,-0.003655
bs_Share Issued,0.999833,1.000000,0.564441,0.168477,0.088441,0.105381,0.121882,0.088416,0.053459,0.088871,...,0.075247,0.053961,0.607544,0.476576,0.903481,0.102542,0.291502,0.472812,0.098776,-0.003917
bs_Net Debt,0.563725,0.564441,1.000000,0.997912,0.684656,0.991991,-0.975023,0.684528,0.653954,0.949582,...,0.563386,0.477271,0.115619,0.255366,0.928694,0.595232,0.797028,0.052567,0.308717,0.739395
bs_Total Debt,0.175143,0.168477,0.997912,1.000000,0.372486,0.507912,0.269541,0.372479,0.482158,0.406292,...,0.273304,0.942352,0.208457,0.016307,0.927750,0.900369,0.587453,0.275057,0.360080,-0.264686
bs_Tangible Book Value,0.088440,0.088441,0.684656,0.372486,1.000000,0.988482,0.986641,1.000000,0.941486,0.999194,...,0.798990,0.998198,0.452595,0.496304,0.670115,0.997680,0.406393,0.465416,0.995338,-0.966476
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fin_Amortization,0.103441,0.102542,0.595232,0.900369,0.997680,0.998193,0.993923,0.997680,0.971083,0.997756,...,0.806548,0.998793,0.193777,0.473874,0.980554,1.000000,0.812073,0.007238,0.994587,-0.977771
fin_Other Special Charges,0.295628,0.291502,0.797028,0.587453,0.406393,0.473757,0.349680,0.406377,0.471517,0.424494,...,0.267825,0.491833,0.822467,0.047402,-0.587669,0.812073,1.000000,0.062098,0.390914,-0.177760
fin_Impairment Of Capital Assets,0.473271,0.472812,0.052567,0.275057,0.465416,0.449228,0.511509,0.465412,0.156671,0.463212,...,0.205080,0.542938,0.940721,0.094569,0.075252,0.007238,0.062098,1.000000,0.567458,-0.111602
fin_Selling And Marketing Expense,0.099196,0.098776,0.308717,0.360080,0.995338,0.982443,0.977209,0.995340,0.941204,0.994521,...,0.796231,0.992842,0.670255,0.622428,0.205309,0.994587,0.390914,0.567458,1.000000,-0.972688


In [ ]:
#Now to train the model ...

models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss'),
    'SVM': SVC(random_state=RANDOM_STATE, probability=True),
    'KNN': KNeighborsClassifier()
    'decision_tree': DecisionTreeClassifier(random_state=RANDOM_STATE)
}
